# STEP Files

Open every tracked STEP artifact in VS Code with the OCP CAD Viewer extension.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import subprocess
import sys

import build123d as bd
from ocp_vscode import show


@dataclass(frozen=True)
class StepArtifact:
    label: str
    step_path: Path
    generator_source: Path
    metadata_path: Path | None = None
    generated: bool = False


def require_repo_root() -> Path:
    result = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    )
    root_text = result.stdout.strip()
    if not root_text:
        raise RuntimeError("git rev-parse --show-toplevel returned empty stdout")
    repo_root = Path(root_text).resolve()
    pyproject_path = repo_root / "pyproject.toml"
    if not pyproject_path.is_file():
        raise FileNotFoundError(f"repo root is missing pyproject.toml: {pyproject_path}")
    return repo_root


def require_file(repo_root: Path, repo_relative_path: Path) -> Path:
    absolute_path = repo_root / repo_relative_path
    if not absolute_path.is_file():
        raise FileNotFoundError(f"registered path is missing: {repo_relative_path}")
    return absolute_path


def ensure_generated_step(repo_root: Path, artifact: StepArtifact) -> Path:
    step_path = repo_root / artifact.step_path
    if artifact.metadata_path is None:
        return require_file(repo_root, artifact.step_path)
    metadata_path = repo_root / artifact.metadata_path
    if not step_path.is_file() or not metadata_path.is_file():
        subprocess.run(
            [
                sys.executable,
                str(repo_root / artifact.generator_source),
                "--output-step",
                str(step_path),
                "--metadata",
                str(metadata_path),
                "--seed",
                "0",
            ],
            cwd=repo_root,
            check=True,
        )
    return step_path


REPO_ROOT = require_repo_root()

STEP_ARTIFACTS = (
    StepArtifact(
        label="type2 non-model objects",
        step_path=Path("examples/type2/artifacts/type2_non_model_objects.step"),
        generator_source=Path("examples/type2/generate_non_model_step.py"),
    ),
    StepArtifact(
        label="tx rect/void modeled coil",
        step_path=Path("run/step/tx_rect_void_coil.step"),
        metadata_path=Path("run/step/tx_rect_void_coil.metadata.json"),
        generator_source=Path("entry/export_tx_rect_void_step.py"),
        generated=True,
    ),
)

TRACKED_STEP_ARTIFACTS = tuple(artifact for artifact in STEP_ARTIFACTS if not artifact.generated)
GENERATED_STEP_ARTIFACTS = tuple(artifact for artifact in STEP_ARTIFACTS if artifact.generated)


In [2]:
tracked_result = subprocess.run(
    ["git", "ls-files", "*.step", "*.stp"],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
TRACKED_STEP_PATHS = tuple(Path(line) for line in tracked_result.stdout.splitlines() if line)
REGISTERED_STEP_PATHS = tuple(artifact.step_path for artifact in TRACKED_STEP_ARTIFACTS)

unregistered_step_paths = tuple(sorted(set(TRACKED_STEP_PATHS) - set(REGISTERED_STEP_PATHS), key=str))
stale_registered_step_paths = tuple(sorted(set(REGISTERED_STEP_PATHS) - set(TRACKED_STEP_PATHS), key=str))
if unregistered_step_paths or stale_registered_step_paths:
    raise RuntimeError(
        "STEP notebook registry mismatch: "
        f"unregistered={unregistered_step_paths}, stale={stale_registered_step_paths}"
    )

for artifact in TRACKED_STEP_ARTIFACTS:
    require_file(REPO_ROOT, artifact.step_path)
    require_file(REPO_ROOT, artifact.generator_source)
for artifact in GENERATED_STEP_ARTIFACTS:
    require_file(REPO_ROOT, artifact.generator_source)

print("available STEP artifacts:")
for index, artifact in enumerate(STEP_ARTIFACTS):
    suffix = "generated" if artifact.generated else "tracked"
    print(f"{index}: {artifact.label} [{suffix}] -> {artifact.step_path}")


available STEP artifacts:
0: type2 non-model objects [tracked] -> examples/type2/artifacts/type2_non_model_objects.step
1: tx rect/void modeled coil [generated] -> run/step/tx_rect_void_coil.step


## Select and show STEP

Set `SELECTED_STEP_INDEX` to one of the printed indices above, then run this single viewer cell.

In [4]:
SELECTED_STEP_INDEX = 0

selected_artifact = STEP_ARTIFACTS[SELECTED_STEP_INDEX]
selected_step_path = ensure_generated_step(REPO_ROOT, selected_artifact)
selected_step = bd.import_step(selected_step_path)
show(
    selected_step,
    names=[selected_artifact.label],
)


++++++c
